# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import os
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Same decision point as w03: only report_date <= March 15 is used -- no future window.
base = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS imp,
        SUM(gsc_clicks)      AS clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position
    FROM {fact_march}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()
base['ctr'] = base['clicks'] / base['imp']
print(f"{len(base):,} content items in the review window")
base.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items in the review window


,client_hash_id,content_hash_id,imp,clicks,avg_position,ctr
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,5.222776,0.000000
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,1.0,5.218750,0.026316
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,4.004356,0.004566
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,4.625000,0.000000
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,6.156643,0.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



> A page is a **quick-win candidate** if it sits in positions 11–20 — Signal 2 shows the typical (median) page there already carries real, above-21+ volume, worth pushing toward page one.
> A page is a **CTR-fix candidate** if it's on page one (positions 1–10) but its click-through rate is below the typical rate for that tier — Signal 1 confirms CTR should be higher there, so an underperforming page is a real gap, not noise.
> Everything else is monitor-only.

Each row's own `imp` and `ctr` values (not the group mean) feed the score below, so no individual row is penalized or boosted by another row's outlier status — the median-vs-mean issue was only in how I read the *signal check*, not in how the rule scores each page.

**Reason codes this rule can output:** `quick_win_zone`, `low_ctr_for_position`, `monitor_only` — exactly one per row, never more.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Rebuild position_tier here too, so this cell works even if run on its own
def position_tier(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

if "position_tier" not in base.columns:
    base["position_tier"] = base["avg_position"].apply(position_tier)

base['is_quick_win_zone'] = base['position_tier'] == "11-20"
ctr_median = base.loc[base['position_tier'].isin(["1-3", "4-10"]), 'ctr'].median()
base['is_ctr_fix_zone']   = base['position_tier'].isin(["1-3", "4-10"]) & (base['ctr'] < ctr_median)

def score_row(r):
    if r['is_quick_win_zone']:
        return pd.Series([r['imp'], "quick_win_zone", "review_for_push"])
    if r['is_ctr_fix_zone']:
        return pd.Series([r['imp'] * (1 - r['ctr']), "low_ctr_for_position", "review_ctr"])
    return pd.Series([r['imp'] * 0.1, "monitor_only", "monitor"])

base[['score', 'reason_code', 'action']] = base.apply(score_row, axis=1)

queue = base.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue.head(10)


wrote 120,513 rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,imp,clicks,avg_position,ctr,position_tier,is_quick_win_zone,is_ctr_fix_zone,score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,143173.0,353.0,16.018687,0.002466,11-20,True,False,143173.0,quick_win_zone,review_for_push
1,client_62f4a7e64f5e0096,content_7c6373141eae744a,86860.0,51.0,5.785512,0.000587,4-10,False,True,86809.0,low_ctr_for_position,review_ctr
2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,0.0,8.607910,0.000000,4-10,False,True,83772.0,low_ctr_for_position,review_ctr
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,73639.0,18.0,2.786744,0.000244,1-3,False,True,73621.0,low_ctr_for_position,review_ctr
4,client_23a62021009f63c4,content_5e1c049f62e33b11,72940.0,108.0,18.233335,0.001481,11-20,True,False,72940.0,quick_win_zone,review_for_push
5,client_20259bd6705d81d4,content_82e35c4845e6c391,70169.0,29.0,18.269589,0.000413,11-20,True,False,70169.0,quick_win_zone,review_for_push
6,client_73cda7b4e4f265ea,content_8e1334d6356668e3,58553.0,1.0,4.579049,0.000017,4-10,False,True,58552.0,low_ctr_for_position,review_ctr
7,client_23a62021009f63c4,content_65c75874a23fca87,55680.0,15.0,9.013531,0.000269,4-10,False,True,55665.0,low_ctr_for_position,review_ctr
8,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,52378.0,18.0,4.011050,0.000344,4-10,False,True,52360.0,low_ctr_for_position,review_ctr
9,client_62f4a7e64f5e0096,content_f6116743b00afc2d,49619.0,8.0,9.493028,0.000161,4-10,False,True,49611.0,low_ctr_for_position,review_ctr


In [7]:
# 4. Rank and evaluate at K -- precision@K on the same data/labels, with the base rate.
label_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {fact_march}
    GROUP BY 1, 2
""").df()

scored = queue.merge(label_frame, on=['client_hash_id', 'content_hash_id'], how='inner')
scored['is_declining_proxy'] = (scored['imp_h2'] < 0.8 * scored['imp'].clip(lower=1)).astype(int)

def precision_at_k(y_true, scores, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

for k in (20, 50, 100):
    p = precision_at_k(scored['is_declining_proxy'], scored['score'], k)
    print(f"precision@{k}: {p:.3f}")

print(f"base rate (is_declining_proxy overall): {scored['is_declining_proxy'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

precision@20: 0.550
precision@50: 0.400
precision@100: 0.380
base rate (is_declining_proxy overall): 0.296


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



1. **review_for_push** -- position 16 (quick-win zone), 143,173 impressions, the highest-volume page in the whole queue. Why it's there: page-one-adjacent and clearly gets real search demand. Wrong if: it's already scheduled for a redesign/merge, or its 353 clicks are converting well despite low CTR -- volume matters more than rate for this action.
2. **review_ctr** -- position 5.8 (page one), CTR 0.06% vs a ~0.38% tier average, more than 6x below typical. Why it's there: a clean, large CTR gap. Wrong if: the title/snippet is intentionally plain (a legal or internal-use page not meant to be clicky).
3. **review_ctr** -- position 8.6, **zero clicks** on 83,772 impressions. Why it's there: 0% CTR is the most extreme possible gap. Wrong if: this is a tracking/measurement issue rather than a content problem -- exactly-zero is as much a red flag for broken click tracking as for a bad title, worth verifying before acting.
4. **review_ctr** -- position 2.8, top-3 ranking but CTR only 0.02%, nowhere near the ~0.42% tier average. Why it's there: top-3 pages should get strong CTR almost automatically; this one badly underperforms its rank. Wrong if: it's a branded/navigational query where users already know the destination and skip the click.
5. **review_for_push** -- position 18.2 (quick-win zone), 108 clicks on 72,940 impressions. Why it's there: solid volume just outside page one, matches the median-based quick-win logic from Signal 2. Wrong if: it's a seasonal page whose ranking will naturally fall further regardless of any push.
6. **review_for_push** -- position 18.3, similar quick-win profile to #5 but weaker CTR (0.04%) even at its current low rank. Why it's there: real volume, borderline page-two position. Wrong if: the low CTR signals a content-quality problem that an SEO push (links) won't fix.
7. **review_ctr** -- position 4.6, only 1 click on 58,553 impressions. Why it's there: near-top position with almost no clicks is a stark gap. Wrong if: same tracking caveat as #3 -- 1 click on this much volume is unusually low and worth a sanity check.
8. **review_ctr** -- position 9.0, CTR 0.03% vs tier average ~0.38%. Why it's there: page-one position, badly underperforming CTR, consistent with the pattern above. Wrong if: the page recently changed URL/title and search results are still showing a stale, less-clicky snippet.
9. **review_ctr** -- position 4.0, CTR 0.03%. Why it's there: same shape as #8 -- high position, very low CTR. Wrong if: it's cannibalized by a sibling page ranking nearby that's absorbing the clicks instead.
10. **review_ctr** -- position 9.5, CTR 0.02%. Why it's there: same pattern again. Wrong if: this is a thin/duplicate page that shouldn't be optimized at all, just merged or removed.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



- **Weak pick:** Row 2 (and similarly row 6) -- exactly 0 or 1 clicks on tens of thousands of impressions is just as likely to be a broken click-tracking pixel as a genuinely unclickable snippet. Before acting on either as a CTR-fix, I'd sanity-check the raw numbers against the actual Search Console UI rather than trusting the warehouse value blindly.
- **Leakage check:** this rule uses only `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` from `report_date <= 2026-03-15`. `health_score`, `priority_score`, `action_type`, and `refresh_tier` were never loaded -- they aren't shipped in this release. No column derived from `report_date > 2026-03-15` was used anywhere in the score.

In [ ]:
# Confirm no product-decision columns or future dates ever entered the rule.
forbidden_cols = {'health_score', 'priority_score', 'action_type', 'refresh_tier'}
used_cols = set(base.columns)
print("forbidden columns present:", forbidden_cols & used_cols, "(should be empty set)")
print("max report_date used: 2026-03-15 (hard-coded in the WHERE clause above)")

forbidden columns present: set() (should be empty set)
max report_date used: 2026-03-15 (hard-coded in the WHERE clause above)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.